# 07 — Đánh giá SVM đã khóa trên test

Notebook này đọc một cấu hình duy nhất từ bước 06. Ảnh test được xử lý bằng đúng biểu diễn đã chọn; PCA, nếu dùng, chỉ được transform bằng đối tượng đã fit trên train. Không chọn lại feature hoặc tham số trên test.


In [ ]:
# KAGGLE_INPUT_SETUP: tìm code và DATA trong các Dataset đã gắn.
from pathlib import Path
import os

kaggle_input = Path('/kaggle/input')
dataset_roots = []
if kaggle_input.is_dir():
    dataset_roots.extend(kaggle_input.glob('datasets/*/*'))
    dataset_roots.extend(kaggle_input.glob('*'))
dataset_roots = [p for p in dataset_roots if p.is_dir()]

project_hint = os.environ.get('CS231_PROJECT_ROOT')
project_candidates = ([Path(project_hint)] if project_hint else
                      [Path.cwd(), *Path.cwd().parents, *dataset_roots])
project_matches = sorted({p.resolve() for p in project_candidates
                          if (p / 'src/SVM/pipeline.py').is_file()})
if len(project_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset code chứa src/SVM/pipeline.py; '
        'nếu có nhiều bản, đặt CS231_PROJECT_ROOT. '
        f'Tìm thấy: {project_matches}'
    )
project = project_matches[0]

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = ([Path(data_hint)] if data_hint else
                   [project / 'DATA', *(p / 'DATA' for p in dataset_roots)])
data_matches = sorted({p.resolve() for p in data_candidates
                       if (p / 'train/10000').is_dir()})
if len(data_matches) != 1:
    raise FileNotFoundError(
        'Cần gắn đúng Dataset DATA chứa DATA/train/10000; '
        'nếu có nhiều bản, đặt CS231_DATA_ROOT. '
        f'Tìm thấy: {data_matches}'
    )
data = data_matches[0]

os.environ['CS231_PROJECT_ROOT'] = str(project)
os.environ['CS231_DATA_ROOT'] = str(data)
print('Code:', project, (project / 'src/SVM/pipeline.py').is_file())
print('DATA:', data, (data / 'train/10000').is_dir())


In [ ]:
PREVIOUS_ARTIFACT = 'selection/locked_svm_config.json'
# PREVIOUS_NOTEBOOK_OUTPUT: khôi phục toàn bộ kết quả của notebook ngay trước đó.
from pathlib import Path
from shutil import copytree
import os

if Path('/kaggle/working').is_dir():
    notebook_root = Path('/kaggle/input/notebooks')
    attached = sorted(p for p in notebook_root.glob('*/*') if p.is_dir())
    if len(attached) != 1:
        raise FileNotFoundError(
            'Hãy gắn đúng một Notebook Output của bước ngay trước đó. '
            f'Đang thấy: {attached}'
        )
    configs = [p for p in attached[0].rglob('experiment_config.json')
               if p.parent.name == 'inputs' and p.parent.parent.name == 'results_224']
    if len(configs) != 1:
        raise FileNotFoundError(
            'Notebook Output đã gắn phải chứa results_224/inputs/experiment_config.json. '
            f'Tìm thấy: {configs}'
        )
    source = configs[0].parents[1]
    expected = source / PREVIOUS_ARTIFACT
    if not expected.is_file():
        raise FileNotFoundError(
            f'Output không phải bản chạy thành công của bước ngay trước: thiếu {expected}'
        )
    target = Path(os.environ.get('CS231_SVM_RESULTS_ROOT',
                                 '/kaggle/working/CS231_SVM/results_224'))
    copytree(source, target, dirs_exist_ok=True)
    for relative in ('inputs/train_manifest.csv', 'inputs/dev_manifest.csv'):
        if not (target / relative).is_file():
            raise FileNotFoundError(f'Output thiếu {target / relative}')
    print('Đã chép:', source, '→', target)
else:
    print('Chạy local: dùng results_224 hiện có trong repository.')


In [ ]:
from pathlib import Path
import os, sys

# Có thể chỉ rõ ba đường dẫn bằng biến môi trường khi gắn repository/dataset trên Kaggle.
root_hint = os.environ.get('CS231_PROJECT_ROOT')
root_candidates = [Path(root_hint)] if root_hint else [Path.cwd(), *Path.cwd().parents]
if Path('/kaggle/input').is_dir():
    root_candidates += list(Path('/kaggle/input').glob('*'))
    root_candidates += list(Path('/kaggle/input').glob('*/*'))
PROJECT_ROOT = next((p.resolve() for p in root_candidates
                     if (p / 'src/SVM/pipeline.py').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Không tìm thấy repository. Đặt CS231_PROJECT_ROOT.')
sys.path.insert(0, str(PROJECT_ROOT))

data_hint = os.environ.get('CS231_DATA_ROOT')
data_candidates = [Path(data_hint)] if data_hint else [PROJECT_ROOT / 'DATA']
if not data_hint and Path('/kaggle/input').is_dir():
    data_candidates += list(Path('/kaggle/input').glob('*/DATA'))
    data_candidates += list(Path('/kaggle/input').glob('*/*/DATA'))
DATA_ROOT = next((p.resolve() for p in data_candidates
                  if (p / 'train/10000').is_dir()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Không tìm thấy DATA/train/10000. Đặt CS231_DATA_ROOT.')

result_hint = os.environ.get('CS231_SVM_RESULTS_ROOT')
if result_hint:
    RESULTS_ROOT = Path(result_hint).resolve()
elif Path('/kaggle/working').is_dir():
    RESULTS_ROOT = Path('/kaggle/working/CS231_SVM/results_224')
else:
    RESULTS_ROOT = PROJECT_ROOT / 'src/SVM/results_224'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Repository:', PROJECT_ROOT)
print('DATA:', DATA_ROOT)
print('Kết quả:', RESULTS_ROOT)


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from src.SVM.pipeline import (
    CLASSES, FEATURE_FOLDERS, manifest_for_split, extract_matrix, l2,
    save_features, load_features, same_manifest, run_dev_grid, save_dev_result,
    save_json, hsv12, hog_raw,
)
from src.preprocessing import read_rgb224


## 1. Kiểm tra khóa và cache


In [ ]:
from src.SVM.pipeline import sha256
from sklearn.svm import SVC
from time import perf_counter
import pickle
lock_path = RESULTS_ROOT / 'selection/locked_svm_config.json'
lock = json.loads(lock_path.read_text(encoding='utf-8'))
for key, expected in lock['sha256'].items():
    domain, relative = key.split('/', 1)
    root = RESULTS_ROOT if domain == 'results' else PROJECT_ROOT
    path = root / relative
    if not path.is_file() or sha256(path) != expected:
        raise ValueError(f'File đã khóa bị thiếu hoặc thay đổi: {key}')
feature_id = lock['feature_id']
feature_dir = RESULTS_ROOT / lock['feature_folder']
X_train, y_train, train_manifest = load_features(feature_dir, 'train')
if X_train.shape != (12494, lock['feature_dim']):
    raise ValueError('Shape train không khớp khóa')
OUTPUT = RESULTS_ROOT / 'test_final'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Phương án đã khóa:', feature_id, lock['svm'])


## 2. Lập manifest test và trích feature

Nhánh HSV12 và HOG dùng cùng hàm của train/dev. Với fusion, các khối được L2 riêng trước khi nối. Ma trận test được lưu để phân tích lỗi, nhưng không dùng để fit PCA hoặc chọn tham số.


In [ ]:
test_manifest = manifest_for_split(DATA_ROOT, 'test')
y_test = test_manifest.label_vnd.to_numpy(dtype=np.int64)
start = perf_counter()
hsv_part = extract_matrix(test_manifest, DATA_ROOT, 'HSV12') if 'HSV12' in feature_id else None
hog_part = extract_matrix(test_manifest, DATA_ROOT, 'HOG_RAW') if 'HOG' in feature_id else None
if hog_part is not None and 'PCA' in feature_id:
    with (RESULTS_ROOT / 'hog_pca/pca.pkl').open('rb') as stream:
        pca = pickle.load(stream)
    hog_part = pca.transform(hog_part)
if hog_part is not None:
    hog_part = l2(hog_part)
if feature_id == 'HSV12':
    X_test = hsv_part
elif feature_id in ('HOG', 'HOG_PCA'):
    X_test = hog_part
elif feature_id in ('HSV12_HOG', 'HSV12_HOG_PCA'):
    X_test = np.concatenate([hsv_part, hog_part], axis=1).astype(np.float32)
else:
    raise ValueError('Biểu diễn chưa được hỗ trợ: ' + feature_id)
feature_seconds = perf_counter() - start
if X_test.shape != (len(test_manifest), lock['feature_dim']) or not np.isfinite(X_test).all():
    raise ValueError('Feature test sai shape hoặc có giá trị không hữu hạn')
print('Test feature:', X_test.shape, X_test.dtype, 'giây:', feature_seconds)


## 3. Fit SVC trên train và dự đoán test

`SVC` học ranh giới từ train; dev chỉ chọn cấu hình. Accuracy, Macro-F1 và MAE theo VND được tính bằng hàm chung của dự án trên toàn bộ test.


In [ ]:
model = SVC(**lock['svm'])
start = perf_counter()
model.fit(X_train, y_train)
fit_seconds = perf_counter() - start
start = perf_counter()
y_pred = model.predict(X_test)
predict_seconds = perf_counter() - start
from src.evaluation.value_metrics import evaluate_classification
scores = evaluate_classification(y_test, y_pred)
summary = pd.DataFrame([{'feature_id': feature_id, 'kernel': lock['svm']['kernel'],
                         'C': lock['svm']['C'], 'gamma': lock['svm']['gamma'],
                         'n_test': len(y_test), 'accuracy': scores['accuracy'],
                         'macro_f1': scores['macro_f1'], 'mae_vnd': scores['mae_vnd']}])
display(summary)
print('Đúng:', int(np.trace(scores['confusion_matrix'])), '/', len(y_test))


## 4. Ma trận nhầm lẫn, bảng lớp và cặp lỗi

MAE chia tổng chênh lệch tuyệt đối cho toàn bộ 1.553 ảnh, kể cả ảnh đúng góp 0. Đây là chênh lệch mệnh giá danh nghĩa, không phải thiệt hại giao dịch.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
cm = np.asarray(scores['confusion_matrix'])
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    ax=ax, cmap='Blues', values_format='d', xticks_rotation=45)
ax.set_title('SVM đã khóa — test')
plt.tight_layout()
plt.savefig(OUTPUT / 'confusion_matrix_test.png', dpi=180)
plt.show()
display(pd.DataFrame(scores['per_class'])[
    ['denomination_vnd', 'support', 'precision', 'recall', 'f1']])
error_columns = ['true_vnd', 'predicted_vnd', 'count', 'gap_vnd', 'total_gap_vnd']
error_pairs = pd.DataFrame(scores['error_pairs'], columns=error_columns)
display(error_pairs.sort_values('total_gap_vnd', ascending=False).head(5))


## 5. Lưu kết quả có thể truy nguồn


In [ ]:
predictions = test_manifest[['relative_path']].copy()
predictions['y_true_vnd'] = y_test
predictions['y_pred_vnd'] = y_pred
predictions['correct'] = y_test == y_pred
predictions['gap_vnd'] = np.abs(y_test - y_pred)
predictions.to_csv(OUTPUT / 'predictions_test.csv', index=False)
test_manifest.to_csv(OUTPUT / 'test_manifest.csv', index=False)
summary.to_csv(OUTPUT / 'summary_test.csv', index=False)
pd.DataFrame(scores['per_class']).to_csv(OUTPUT / 'per_class_test.csv', index=False)
error_pairs.to_csv(OUTPUT / 'error_pairs_test.csv', index=False)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(OUTPUT / 'confusion_matrix_test.csv')
np.save(OUTPUT / 'X_test.npy', X_test)
save_json(OUTPUT / 'metrics_test.json', scores)
(OUTPUT / 'locked_svm_config_used.json').write_bytes(lock_path.read_bytes())
save_json(OUTPUT / 'run_info.json', {
    'lock_sha256': sha256(lock_path), 'feature_seconds': feature_seconds,
    'fit_seconds': fit_seconds, 'predict_seconds': predict_seconds,
    'n_support_vectors': int(model.n_support_.sum()),
    'train_samples': len(y_train), 'test_samples': len(y_test),
    'test_used_for_tuning': False,
})
print('Đã lưu:', OUTPUT)
